# 案例 2 · 预测基线与评估 (Baselines & Evaluation)

对应讲义 [第 7 讲](https://jiangyou2025.github.io/kun/course/07/) 与 [第 8 讲](https://jiangyou2025.github.io/kun/course/08/)。

本案例演示一条完整的预测工作流：
1. 按时间切分 train / test；
2. 三个基线：朴素、季节性朴素、移动平均；
3. 用 MAE / RMSE / MAPE 评估；
4. 滚动回测 (rolling backtest)。

> 任何复杂模型（包括 KUN）都必须先打败这些基线，否则就是模型有问题。

> 依赖：`numpy`, `pandas`, `matplotlib`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(0)
n = 730; t = np.arange(n)
y = 20 + 0.05*t + 4*np.sin(2*np.pi*t/7) + 10*np.sin(2*np.pi*t/365.25) + np.random.normal(0, 2, n)
idx = pd.date_range('2022-01-01', periods=n, freq='D')
s = pd.Series(y, index=idx, name='value')
s.tail()

## 1. 按时间切分
预测时域 `H = 30`：用最后 30 天作为 test，其余作为 train。**绝不随机打乱。**

In [ ]:
H = 30
train, test = s.iloc[:-H], s.iloc[-H:]
print('train:', len(train), ' test:', len(test))

## 2. 三个基线模型

In [ ]:
def naive(train, H):
    return np.repeat(train.iloc[-1], H)

def seasonal_naive(train, H, m=7):
    last = train.iloc[-m:].values
    return np.array([last[i % m] for i in range(H)])

def moving_average(train, H, w=30):
    return np.repeat(train.iloc[-w:].mean(), H)

## 3. 评价指标

In [ ]:
def mae(a, f):  return np.mean(np.abs(a - f))
def rmse(a, f): return np.sqrt(np.mean((a - f)**2))
def mape(a, f): return 100 * np.mean(np.abs((a - f) / a))

a = test.values
preds = {
    'naive':          naive(train, H),
    'seasonal_naive': seasonal_naive(train, H, 7),
    'moving_avg':     moving_average(train, H, 30),
}
rows = [[name, mae(a, f), rmse(a, f), mape(a, f)] for name, f in preds.items()]
res = pd.DataFrame(rows, columns=['model', 'MAE', 'RMSE', 'MAPE']).set_index('model')
res.round(3)

## 4. 可视化预测 vs 实际

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(train.index[-90:], train.values[-90:], label='train (tail)')
plt.plot(test.index, a, color='black', label='actual')
for name, f in preds.items():
    plt.plot(test.index, f, '--', label=name)
plt.legend(); plt.title('Baselines vs actual')
plt.tight_layout(); plt.show()

## 5. 滚动回测 (Rolling backtest)
一个测试段的分数可能靠运气。滚动回测在多个时间起点上重复评估，给出误差的分布。

In [ ]:
def backtest_seasonal(s, H=30, m=7, folds=6):
    errs = []
    for k in range(folds, 0, -1):
        cut = len(s) - k*H
        if cut <= m:
            continue
        tr, te = s.iloc[:cut], s.iloc[cut:cut+H]
        f = seasonal_naive(tr, len(te), m)
        errs.append(mae(te.values, f))
    return np.array(errs)

errs = backtest_seasonal(s)
print('MAE per fold:', errs.round(3))
print('mean MAE    :', errs.mean().round(3))

## 小结
- 季节性朴素通常是最难打败的简单基线。
- 永远把模型分数**和基线对比**，并用滚动回测取得稳健的估计。
- 下一步见 [案例 3](03_kun_style_window_forecast.ipynb)：用滑动窗口 + 神经网络（KUN 思想）做直接多步预测。